# The categories $\mathbf{Span}$ and $\mathbf{CoSpan}$

A small companion to `tract.span` and `tract.cospan`. The category $\mathbf{Span}$ has flat tuples of positive integers as objects, and a morphism $U \to V$ is a span

$$U \xleftarrow{\;b\;} X \xrightarrow{\;f\;} V$$

whose backward (left) leg $b\colon X \twoheadrightarrow U$ is a $\mathbf{Fact}$ morphism and whose forward (right) leg $f\colon X \to V$ is a $\mathbf{Tuple}$ morphism, sharing the apex $X$. The apex is thus a flat refinement of $U$ equipped with a tuple morphism into $V$.

In [1]:
from tract import Fact_morphism, Span_morphism, Tuple_morphism

## Constructing spans

In [2]:
# (12,) <--((4,3),)-- (4,3) --(1,2)--> (4,3,2)
b1 = Fact_morphism(domain=(4, 3), codomain=(12,), modes=((4, 3),))
f1 = Tuple_morphism(domain=(4, 3), codomain=(4, 3, 2), map=(1, 2))
s1 = Span_morphism(b1, f1)
print(s1)
print("domain:", s1.domain, " apex:", s1.apex, " codomain:", s1.codomain)

(12,) <--((4, 3),)-- (4, 3) --(1, 2)--> (4, 3, 2)
domain: (12,)  apex: (4, 3)  codomain: (4, 3, 2)


## Composition via pullback

To compose $U \xleftarrow{b_1} X \xrightarrow{f_1} V$ with $V \xleftarrow{b_2} Y \xrightarrow{f_2} W$, we pull $f_1$ back along $b_2$ (`Fact_morphism.pullback_with_refinement`), completing the square with a new corner $X'$:

$$\begin{array}{ccccc}
& & X' & \xrightarrow{f_1'} & Y & \xrightarrow{f_2} & W\\
& & \downarrow r & & \downarrow b_2\\
U & \xleftarrow{b_1} & X & \xrightarrow{f_1} & V
\end{array}$$

The composite is $U \xleftarrow{b_1 \circ r} X' \xrightarrow{f_2 \circ f_1'} W$. Below, the second span refines the entry $4$ of $V$ into $(2,2)$, so the pullback refines the apex accordingly.

In [3]:
# (4,3,2) <--((2,2),(3,),(2,))-- (2,2,3,2) --(1,2,0,3)--> (2,2,2)
b2 = Fact_morphism((2, 2, 3, 2), (4, 3, 2), ((2, 2), (3,), (2,)))
f2 = Tuple_morphism((2, 2, 3, 2), (2, 2, 2), (1, 2, 0, 3))
s2 = Span_morphism(b2, f2)

composite = s1.compose(s2)   # diagrammatic order: s1 then s2
print(composite)
print("apex refined from", s1.apex, "to", composite.apex)

(12,) <--((2, 2, 3),)-- (2, 2, 3) --(1, 2, 0)--> (2, 2, 2)
apex refined from (4, 3) to (2, 2, 3)


## Identities and category laws

In [4]:
idV = Span_morphism.identity(s1.codomain)
print("identity on", s1.codomain, ":", idV)
print("unit laws hold:",
      s1.compose(idV) == s1 and Span_morphism.identity(s1.domain).compose(s1) == s1)

identity on (4, 3, 2) : (4, 3, 2) <--((4,), (3,), (2,))-- (4, 3, 2) --(1, 2, 3)--> (4, 3, 2)
unit laws hold: True


## Remark: strictness

In general, composition in a span category is defined via a *chosen* pullback and is associative only up to canonical isomorphism of spans (so one usually gets a bicategory). Here, the concrete pullback — substituting each apex entry by its refining block, with basepoint entries passing through untouched — is strictly functorial: composition is associative and unital **on the nose**, so $\mathbf{Span}$ as implemented is a genuine 1-category with no isomorphism bookkeeping. The test suite (`tests/span_tests.py`) verifies strict associativity, strict unitality, and the interchange law with the legwise sum $\oplus$ over randomized inputs.

## The category $\mathbf{CoSpan}$

Dually, a morphism $U \to V$ in $\mathbf{CoSpan}$ is a cospan

$$U \xrightarrow{\;f\;} X \xleftarrow{\;b\;} V$$

whose forward (left) leg $f\colon U \to X$ is a $\mathbf{Tuple}$ morphism and whose backward (right) leg $b\colon X \twoheadrightarrow V$ is a $\mathbf{Fact}$ morphism out of the nadir $X$ (pointing backward, from the nadir to $V$). In both categories the $\mathbf{Fact}$ leg has the apex/nadir as its domain: the middle object is a flat refinement of the boundary object it points to.

In [5]:
from tract import CoSpan_morphism

# (4,3) --(1,2)--> (4,3,2) <--((4,3),(2,))-- (12,2)
f1 = Tuple_morphism((4, 3), (4, 3, 2), (1, 2))
b1 = Fact_morphism((4, 3, 2), (12, 2), ((4, 3), (2,)))
c1 = CoSpan_morphism(f1, b1)
print(c1)
print("domain:", c1.domain, " nadir:", c1.nadir, " codomain:", c1.codomain)

(4, 3) --(1, 2)--> (4, 3, 2) <--((4, 3), (2,))-- (12, 2)
domain: (4, 3)  nadir: (4, 3, 2)  codomain: (12, 2)


Composition is dual: to compose $U \xrightarrow{f_1} X \xleftarrow{b_1} V$ with $V \xrightarrow{f_2} Y \xleftarrow{b_2} W$, we push $f_2$ forward along $b_1$ (`Fact_morphism.pushforward_with_refinement`), completing the square with a new corner $Y'$, and compose the legs:

$$U \xrightarrow{\;f_2' \circ f_1\;} Y' \xleftarrow{\;b_2 \circ r\;} W.$$

Below, the first cospan's nadir refines $12$ into $(4,3)$, so the pushforward refines the composite nadir accordingly.

In [6]:
# (12,2) --(1,2)--> (12,2,5) <--((12,),(2,5))-- (12,10)
f2 = Tuple_morphism((12, 2), (12, 2, 5), (1, 2))
b2 = Fact_morphism((12, 2, 5), (12, 10), ((12,), (2, 5)))
c2 = CoSpan_morphism(f2, b2)

composite = c1.compose(c2)   # diagrammatic order: c1 then c2
print(composite)
print("nadir refined from", c2.nadir, "to", composite.nadir)

idV = CoSpan_morphism.identity(c1.codomain)
print("unit laws hold:",
      c1.compose(idV) == c1 and CoSpan_morphism.identity(c1.domain).compose(c1) == c1)

(4, 3) --(1, 2)--> (4, 3, 2, 5) <--((4, 3), (2, 5))-- (12, 10)
nadir refined from (12, 2, 5) to (4, 3, 2, 5)
unit laws hold: True


As with $\mathbf{Span}$, the chosen pushforward makes composition strictly associative and unital, so $\mathbf{CoSpan}$ is a genuine 1-category; `tests/cospan_tests.py` verifies the laws over randomized inputs.